# AI Foundry Key Verification

This notebook verifies that the API keys in the Excel file (`output/team_keys.xlsx`) generated by `deploy_all.sh` are working correctly.

## Authentication Method

This sample uses **API key authentication** (`api-key` header) only.  
It assumes a scenario where external developers access Azure AI Services endpoints using only the Resource Key, without Entra ID.

> **PoC only**: For production environments, using Azure Key Vault + Entra ID (DefaultAzureCredential) is recommended.  
> Keys stored in plaintext in the Excel file are for sample/PoC purposes only — never commit them to Git.

In [ ]:
# --- Dependencies ---
%pip install openpyxl requests --quiet

import openpyxl
import requests
from pathlib import Path

In [ ]:
# --- Configuration ---
# Excel file path (generated after running deploy_all.sh)
EXCEL_PATH = Path("../output/team_keys.xlsx")

# Model deployment name for Chat Completion (e.g., gpt-4o)
CHAT_DEPLOYMENT = "gpt-4o"

# Model deployment name for Embedding (e.g., text-embedding-3-large)
EMBEDDING_DEPLOYMENT = "text-embedding-3-large"

# Azure OpenAI API version
API_VERSION = "2024-12-01-preview"

In [ ]:
# --- Read Excel & Display Teams ---
if not EXCEL_PATH.exists():
    raise FileNotFoundError(
        f"{EXCEL_PATH} not found. Run './scripts/deploy_all.sh' first to generate the key Excel."
    )

wb = openpyxl.load_workbook(EXCEL_PATH, read_only=True)
ws = wb.active

# Parse rows (skip header)
rows = list(ws.iter_rows(min_row=2, values_only=True))
wb.close()

if not rows:
    raise ValueError("Excel file has no data rows.")

# Display available teams/regions
print(f"{'#':<4} {'Team':<12} {'Region':<18} {'Endpoint':<55} {'Models'}")
print("-" * 120)
for idx, row in enumerate(rows):
    team, _sub, _rg, region, endpoint, _k1, _k2, models = row
    print(f"{idx:<4} {team:<12} {region:<18} {endpoint:<55} {models}")

print(f"\nFound {len(rows)} Foundry Resource(s)")

In [ ]:
# --- Select Row & Test Chat Completion ---
# Select the row number to test (refer to the # column in the table above)
SELECTED_ROW = 0

row = rows[SELECTED_ROW]
team, _sub, _rg, region, endpoint, key1, _key2, models = row

print(f"Team: {team} | Region: {region}")
print(f"Endpoint: {endpoint}")
print(f"Models: {models}")
print()

# Chat Completion call
url = f"{endpoint.rstrip('/')}/openai/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
headers = {
    "Content-Type": "application/json",
    "api-key": key1,
}
payload = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say 'Key verification successful!' in one sentence."},
    ],
    "max_tokens": 50,
}

print(f"Calling {CHAT_DEPLOYMENT}...")
resp = requests.post(url, headers=headers, json=payload, timeout=30)
resp.raise_for_status()

result = resp.json()
message = result["choices"][0]["message"]["content"]
usage = result.get("usage", {})

print(f"Response: {message}")
print(f"Tokens \u2014 prompt: {usage.get('prompt_tokens')}, completion: {usage.get('completion_tokens')}, total: {usage.get('total_tokens')}")
print("\n\u2705 Chat Completion key verification successful!")

In [ ]:
# --- Test Embedding ---
# The selected Foundry Resource must have a text-embedding model deployed.
if EMBEDDING_DEPLOYMENT not in (models or ""):
    print(f"\u26a0\ufe0f  '{EMBEDDING_DEPLOYMENT}' model is not deployed in '{team}/{region}'. Skipping.")
else:
    emb_url = f"{endpoint.rstrip('/')}/openai/deployments/{EMBEDDING_DEPLOYMENT}/embeddings?api-version={API_VERSION}"
    emb_payload = {
        "input": "Azure AI Foundry cost governance sample",
    }

    print(f"Calling {EMBEDDING_DEPLOYMENT}...")
    emb_resp = requests.post(emb_url, headers=headers, json=emb_payload, timeout=30)
    emb_resp.raise_for_status()

    emb_result = emb_resp.json()
    embedding = emb_result["data"][0]["embedding"]
    emb_usage = emb_result.get("usage", {})

    print(f"Embedding dimensions: {len(embedding)}")
    print(f"First 5 values: {embedding[:5]}")
    print(f"Tokens \u2014 prompt: {emb_usage.get('prompt_tokens')}, total: {emb_usage.get('total_tokens')}")
    print("\n\u2705 Embedding key verification successful!")

## Next Steps

Key verification is complete. Next:

1. **Check Cost Dashboard**: Azure Portal → each Team Subscription → Application Insights → Workbook to view token usage and costs.
2. **Budget Alert**: When the threshold set by the Terraform-deployed Budget Alert is exceeded, a notification is sent to `alert_email`.
3. **For production migration**: Switch from API keys to Azure Key Vault + Entra ID (`DefaultAzureCredential`) based authentication.

---

For more details, see [README.en.md](../README.en.md).